# 01 — Graph Fundamentals

Notebook 00 introduced routing. This lesson slows down and focuses on the mechanics every later graph uses: state, nodes, explicit updates, reducers, fixed edges, compilation, invocation, and execution traces.

The graph remains deterministic so model behavior cannot distract from execution semantics.

## Read the graph before the code

A fixed edge means the next step is known. Each node adds one trace entry through a reducer.

```mermaid
flowchart LR
    accTitle: Linear Fundamentals Graph
    accDescr: Text is normalized, counted, and summarized in a fixed sequence before the graph ends.

    start([START]) --> normalize[Normalize]
    normalize --> count[Count words]
    count --> summarize[Summarize]
    summarize --> finish([END])
```

## State is the node interface

The typed state declares which values may cross node boundaries. `trace` is annotated with `operator.add`, so each returned list is appended rather than replacing prior events.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class TextState(TypedDict, total=False):
    text: str
    normalized_text: str
    word_count: int
    summary: str
    trace: Annotated[list[str], operator.add]

## Each node owns one computation

A node reads the fields it needs and returns only changed fields. Local variables do not belong in shared state.

In [2]:
def normalize(state: TextState) -> dict:
    cleaned = " ".join(state["text"].strip().split())
    return {"normalized_text": cleaned, "trace": ["normalize"]}


def count_words(state: TextState) -> dict:
    count = len(state["normalized_text"].split())
    return {"word_count": count, "trace": ["count_words"]}


def summarize(state: TextState) -> dict:
    summary = f"Normalized text contains {state['word_count']} words."
    return {"summary": summary, "trace": ["summarize"]}

## Inspect an update before building the graph

Calling a node directly is a useful unit test. The input should remain unchanged; the return value is the proposed update.

In [3]:
sample_state = {"text": "  Graphs   make control explicit.  ", "trace": []}
update = normalize(sample_state)

assert sample_state["text"].startswith("  ")
assert update == {
    "normalized_text": "Graphs make control explicit.",
    "trace": ["normalize"],
}
print(update)

{'normalized_text': 'Graphs make control explicit.', 'trace': ['normalize']}


## Build, connect, and compile

Nodes become executable graph steps only after they are registered and connected. `compile()` creates the runnable graph.

In [4]:
builder = StateGraph(TextState)
builder.add_node("normalize", normalize)
builder.add_node("count_words", count_words)
builder.add_node("summarize", summarize)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "count_words")
builder.add_edge("count_words", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()

## Invoke and inspect merged state

Invocation starts with input state. After each node, the runtime merges its update according to the state schema and reducer annotations.

In [5]:
result = graph.invoke({"text": "  Graphs   make control explicit.  ", "trace": []})

assert result["word_count"] == 4
assert result["trace"] == ["normalize", "count_words", "summarize"]
print(result)

{'text': '  Graphs   make control explicit.  ', 'normalized_text': 'Graphs make control explicit.', 'word_count': 4, 'summary': 'Normalized text contains 4 words.', 'trace': ['normalize', 'count_words', 'summarize']}


## Stream node updates

The final state shows what exists at the end. Update streaming shows which node contributed each change. This is the beginning of graph observability.

In [6]:
for event in graph.stream(
    {"text": "State crosses node boundaries.", "trace": []},
    stream_mode="updates",
):
    print(event)

{'normalize': {'normalized_text': 'State crosses node boundaries.', 'trace': ['normalize']}}
{'count_words': {'word_count': 4, 'trace': ['count_words']}}
{'summarize': {'summary': 'Normalized text contains 4 words.', 'trace': ['summarize']}}


## Takeaways

- State is a cross-node contract, not a bag of local variables.
- Nodes return explicit updates.
- Reducers define how repeated updates merge.
- Edges define execution order.
- Compilation and invocation are separate steps.
- Traces should explain which nodes changed state.

Next: Notebook 02 adds conditional routing and an optional semantic classifier.